# Library

In [1]:
import csv
import time
import json
import os
import asyncio

from search_on_company_site import on_company_site_search, on_company_site_search_save_evaluate
from search_on_nikkei import nikkei_governance_search, nikkei_governance_search_save_evaluate
from search_on_jpx import JPXGovernanceScraper, jpx_governance_search_save_evaluate
from search_governance_fallback import search_governance_fallback, search_governance_fallback_batch
from automation_bot_DFS import AutomationSearchReportDFS

from validator_llm import SearchReportValidator
from searcher_searxng import SearXNGSearch
from search_combine import SearchReportCombine
from automation_bot import AutomationBot, run_bot

validator = SearchReportValidator()
searcher = SearXNGSearch()

# Data

In [2]:
company_info = {
    "6920": {"name": "Lasertec Corporation", "site": "https://www.lasertec.co.jp/"},
    "4063": {"name": "Shin-Etsu Chemical Co., Ltd.", "site": "https://www.shinetsu.co.jp/"},
    "9104": {"name": "Mitsui O.S.K. Lines, Ltd.", "site": "https://www.mol.co.jp/"},
    "2432": {"name": "DeNA Co., Ltd.", "site": "https://dena.com/intl/"},
    "9697": {"name": "CAPCOM CO., LTD.", "site": "https://www.capcom.co.jp/"},
    "5108": {"name": "BRIDGESTONE CORPORATION", "site": "https://www.bridgestone.co.jp/"},
    "4568": {"name": "DAIICHI SANKYO COMPANY, LIMITED", "site": "https://www.daiichisankyo.co.jp/"},
    "1812": {"name": "KAJIMA CORPORATION", "site": "https://www.kajima.co.jp/"},
    "5411": {"name": "JFE Holdings, Inc.", "site": "https://www.jfe-holdings.co.jp/"},
    "7974": {"name": "Nintendo Co., Ltd.", "site": "https://www.nintendo.co.jp/"},
    "7011": {"name": "Mitsubishi Heavy Industries, Ltd.", "site": "https://www.mhi.com/"},
    "8001": {"name": "ITOCHU Corporation", "site": "https://www.itochu.co.jp/"},
    "6952": {"name": "CASIO COMPUTER CO., LTD.", "site": "https://www.casio.co.jp/"},
    "4393": {"name": "Bank of Innovation, Inc.", "site": "https://www.boi.jp/"},
    "8306": {"name": "Mitsubishi UFJ Financial Group, Inc.", "site": "https://www.mufg.jp/"},
    "6857": {"name": "ADVANTEST CORPORATION", "site": "https://www.advantest.com/"},
    "7203": {"name": "TOYOTA MOTOR CORPORATION", "site": "https://www.global.toyota/"},
    "9983": {"name": "FAST RETAILING CO., LTD.", "site": "https://www.fastretailing.com/"},
    "8801": {"name": "Mitsui Fudosan Co., Ltd.", "site": "https://www.mitsuifudosan.co.jp/"},
    "9984": {"name": "SoftBank Group Corp.", "site": "https://www.group.softbank/"},
    "3635": {"name": "KOEI TECMO HOLDINGS CO., LTD.", "site": "https://www.koeitecmo.co.jp/"},
    "4816": {"name": "TOEI ANIMATION CO., LTD.", "site": "https://www.toei-anim.co.jp/"},
    "5020": {"name": "ENEOS Holdings, Inc.", "site": "https://www.hd.eneos.co.jp/"},
    "8058": {"name": "Mitsubishi Corporation", "site": "https://www.mitsubishicorp.com/"},
    "4689": {"name": "LY Corporation", "site": "https://www.lycorp.co.jp/"},
}

In [5]:
STOCK_ID= [
    "6920.T",
    "4063.T",
    "9104.T",
    "2432.T",
    "9697.T",
    "5108.T",
    "4568.T",
    "1812.T",
    "5411.T",
    "7974.T",
    "7011.T",
    "8001.T",
    "6952.T",
    "4393.T",
    "8306.T",
    "6857.T",
    "7203.T",
    "9983.T",
    "8801.T",
    "9984.T",
    "3635.T",
    "4816.T",
    "5020.T",
    "8058.T",
    "4689.T"
]

# A/ Intergrated report

### 1 - 🌐 Search On Company Site

```mermaid
flowchart TD

    A["Company Website – Toyota Search Query: site:global.toyota integrated reports filetype:pdf"]

    A --> B["SearXNG Search (Meta-search Engine)"]

    B --> C["Validate with LLM (Best Report Filter)"]

    C --> D["Url"]
```

In [4]:
# Run with oine stock code
best_url = on_company_site_search(
    searcher,
    validator,
    stock_code="9983.T",
    search_keyword="integrated report",
    result_label="Integrated"
)
print("best_url:", best_url)


Query: site:fastretailing.com filetype:pdf integrated report


2026-03-02 10:36:36,957 - pyserxng.client - INFO - Search completed: 32 results in 2.98s from http://localhost:8888/


Found 32 results

best_url: https://www.fastretailing.com/eng/ir/library/pdf/ar2025_en.pdf


In [ ]:
# Run list Stock Code - save in a csv file
on_company_site_search_save_evaluate(
    searcher,
    validator,
    stock_list=STOCK_ID,
    output_file="integrated_reports_on_company_site.csv",
    search_keyword="integrated report",
    result_label="Integrated"
)

# B/ Corporate governance report

### 1 - 🌐 Search On Company Site

```mermaid
flowchart TD

    A["Company Website – Toyota Search Query: site:global.toyota corporate governance reports filetype:pdf"]

    A --> B["SearXNG Search (Meta-search Engine)"]

    B --> C["Validate with LLM (Best Report Filter)"]

    C --> D["Url"]
```

In [ ]:
# Run with oine stock code
best_url = on_company_site_search(
    searcher,
    validator,
    stock_code="9983.T",
    search_keyword="corporate governance report",
    result_label="Governance"
)
print("best_url:", best_url)

In [ ]:
# Run list Stock Code - save in a csv file
on_company_site_search_save_evaluate(
    searcher,
    validator,
    stock_list=STOCK_ID,
    output_file="corporate_governance_reports_on_company_site.csv",
    search_keyword="corporate governance report",
    result_label="Governance"
)

### 2 - 🗞 Search On Nikkei

```mermaid
flowchart TD

    A["Search on Nikkei Site Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 CORPORATE GOVERNANCE 最終更新日 {name} filetype:pdf"]

    A --> B["SearXNG Search (Meta-search Engine)"]

    B --> C["Validate with LLM (Best Report Filter)"]

    C --> D["Url"]
```

In [ ]:
# Run with oine stock code
best_url = nikkei_governance_search(
    searcher, 
    validator, 
    stock_code = "6920.T"
)
print("best_url:", best_url)

In [ ]:
# Run list Stock Code - save in a csv file
nikkei_governance_search_save_evaluate(
    searcher,
    validator,
    stock_list=STOCK_ID,
    output_file="nikkei_governance_best_results.csv",
    delay=10
)

### 3 - 🏛 Search On JPX

```mermaid
flowchart TD

    A["Japan Exchange Group (JPX) Listed Company Search Database URL: https://www2.jpx.co.jp/tseHpFront/JJK020030Action.do"]

    A --> B["Search by Stock Code (Direct Lookup)"]

    B -->|Playwright Automation| C["Click Corporate Governance Tab"]

    C --> D["Extract Japanese Table (Latest Governance Data)"]

    D --> E["Parse PDF Link & Date (Official Document)"]

    E --> F["Url"]
```

In [7]:
async with JPXGovernanceScraper(headless=True) as jpx_scraper:
    best_url = await jpx_scraper.get_latest_governance(stock_code="6920.T")
    print("best_url:", best_url)

best_url: {'date': '2025/09/29', 'pdf_url': 'https://www2.jpx.co.jp/disc/69200/140120250822545830.pdf'}


In [ ]:
await jpx_governance_search_save_evaluate(
    stock_list=STOCK_ID,
    output_file="jpx_governance.csv",
    headless = True
)

### 4 - 🧩 Search Combine

```mermaid
    flowchart TD

        A[Multiple Search Sources Combined]

        A --> B[Company Site]
        A --> C[Nikkei Site]
        A --> D[JPX Site]

        B --> E["Parse & Normalize Results(Date, URL, Source)"]
        C --> E
        D --> E

        E --> F["Compare & Select Latest(Most Recent Date)"]

        F --> G["Save Best Result to CSV (Highest Quality Source)"]
```

In [8]:
# Search single stock code
combine = SearchReportCombine()
result = await combine.search_single_company(stock_code="6920.T")
print("\nResult:")
print(result)

Processing Lasertec Corporation (6920.T)

Query: site:lasertec.co.jp filetype:pdf corporate governance report


2026-03-02 10:40:24,285 - pyserxng.client - INFO - Search completed: 6 results in 2.42s from http://localhost:8888/


Found 6 results


Query: site:www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3 filetype:pdf CORPORATE GOVERNANCE 最終更新日 Lasertec Corporation


2026-03-02 10:40:34,222 - pyserxng.client - INFO - Search completed: 20 results in 1.90s from http://localhost:8888/


Found 20 results

✅ Found: nikkei_site - 2025-10-23

Result:
{'source': 'nikkei_site', 'url': 'https://www.nikkei.com/markets/ir/irftp/data/tdnr/tdnetg3/20251023/fl8u76/140120251022576962.pdf', 'date': datetime.datetime(2025, 10, 23, 0, 0), 'raw_date': '2025-10-23'}


In [ ]:
governance_combine_search = SearchReportCombine()
await governance_combine_search.process_companies(
    stock_list=STOCK_ID,
    output_file="governance_reports_combined.csv"
)

### 5 - 🛟 Search Fallback

```mermaid
flowchart TD
    A["Start: Search Governance Report"]
    
    A --> B["JPX Search"]
    B --> B1{Is PDF?}
    B1 -->|Yes| G["✅ Return URL + Date<br/>Source: JPX"]
    B1 -->|No/Fail| C["Company Site Search"]
    
    C --> C1{Is PDF?}
    C1 -->|Yes| G
    C1 -->|No/Fail| D["Nikkei Search"]
    
    D --> D1{Is PDF?}
    D1 -->|Yes| G
    D1 -->|No/Fail| H["❌ Return None<br/>All sources failed"]
    
    G --> I[URL]
    H --> I
```


In [ ]:
# Single stock
result = await search_governance_fallback(
    searcher=searcher,
    validator=validator,
    stock_code="6920.T",
    company_site="https://www.lasertec.co.jp/"
)
print(result)  # {'url': '...', 'source': 'jpx', 'success': True, ...}

In [ ]:
# Batch processing
await search_governance_fallback_batch(
    searcher=searcher,
    validator=validator,
    stock_list=STOCK_ID,
    output_file="governance_fallback_results.csv"
)

# C/ Automation bot

```mermaid
    graph LR
        A[Start URL] --> B[Crawl Page]
        B --> C[Extract Links]
        C --> D["LLM Analysis"]
        D --> E{Found PDF?}
        E -->|Yes| F["Return PDF URL"]
        E -->|No| G[Select Next URL]
        G --> B
```

In [4]:
# Search 1 stock code
auto_bot = AutomationBot(headless=True, report_type="Integrated Report")
result = await auto_bot.run_single_company("4393.T")
print(result)

✓ Graph saved successfully: graph.png

🔎 Processing Bank of Innovation,Inc. (4393.T)

Crawling: https://boi.jp

🔎 Processing Bank of Innovation,Inc. (4393.T)

Crawling: https://boi.jp
⚠ Không thấy PDF selector (có thể trang không có)
🔗 Extracted 59 links from https://boi.jp
⚠ Không thấy PDF selector (có thể trang không có)
🔗 Extracted 59 links from https://boi.jp
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://boi.jp/ir/'}
📍 Next URL: https://boi.jp/ir/

Crawling: https://boi.jp/ir/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://boi.jp/ir/'}
📍 Next URL: https://boi.jp/ir/

Crawling: https://boi.jp/ir/
🔗 Extracted 56 links from https://boi.jp/ir/
🔗 Extracted 56 links from https://boi.jp/ir/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://boi.jp/ir/library/'}
📍 Next URL: https://boi.jp/ir/library/

Crawling: https://boi.jp/ir/library/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://boi.jp/ir/library/'}
📍 Next URL: https://boi.jp/ir/library/

Crawling: https://boi

In [ ]:
# Search stock codes
auto_bot = AutomationBot(headless=True, report_type="Integrated Report")
await run_bot(
    auto_bot=auto_bot,
    stock_list=STOCK_ID,
    output_file="integrated_report_results_bot.csv"
)

In [3]:
# company_info = {
#     "6920": {"name": "Lasertec Corporation", "site": "https://www.lasertec.co.jp/"},
# }

# auto_bot = AutomationBot(headless=True, report_type="Corporate Governance Report")

# await run_bot(
#     auto_bot=auto_bot,
#     company_info=company_info,
#     output_file="test.csv"
# )

✓ Graph saved successfully: graph.png

🔎 Processing Lasertec Corporation (6920)

Crawling: https://www.lasertec.co.jp/
🔗 Extracted 116 links from https://www.lasertec.co.jp/
🔗 Extracted 116 links from https://www.lasertec.co.jp/
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/sustainability/governance.html'}
📍 Next URL: https://www.lasertec.co.jp/sustainability/governance.html

Crawling: https://www.lasertec.co.jp/sustainability/governance.html
🤖 LLM: {'next_step': 'continue_find', 'url': 'https://www.lasertec.co.jp/sustainability/governance.html'}
📍 Next URL: https://www.lasertec.co.jp/sustainability/governance.html

Crawling: https://www.lasertec.co.jp/sustainability/governance.html
🔗 Extracted 105 links from https://www.lasertec.co.jp/sustainability/governance.html
🔗 Extracted 105 links from https://www.lasertec.co.jp/sustainability/governance.html
🤖 LLM: {'next_step': 'END', 'url': 'https://ssl4.eir-parts.net/doc/6920/tdnet/2691142/00.pdf'}
✅ Found PDF: htt

# Automation bot - Deep First Search

In [6]:
auto_bot = AutomationSearchReportDFS()

In [4]:
stock_code = "1812.T"
results = await auto_bot.run(stock_code = stock_code)
print(f"{stock_code}: {results['current_url']}")

⚠️ No PDF links found, proceeding with available links.
1812.T: https://www.kajima.co.jp/sustainability/report/2025/pdf/ir_all.pdf


In [ ]:
for stock_code in STOCK_ID:
    results = await auto_bot.run(stock_code = stock_code)
    print(f"{stock_code}: {results['current_url']}")